# InternVL3 Document-Aware Batch Processing

## Clean Implementation - Based on Working Single-Image Pattern

This notebook implements batch processing using the **exact same successful pattern** from `internvl3_document_aware.ipynb` that works perfectly on H200 GPU.

### Key Architecture:
- ✅ **Simple Direct Approach** - No complex handlers or recursion protection
- ✅ **Proven Model Loading** - Uses exact same H200 direct loading that works
- ✅ **Clean Loop Structure** - Process images one by one with working processor
- ✅ **YAML-First Detection** - Same approach as successful single-image notebook


In [ ]:
import gc
import os
import torch

# CRITICAL: Set V100-optimized CUDA memory allocation (from working system)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:64'
print("🔧 CUDA memory allocation configured: max_split_size_mb:64 (V100 optimized)")

# Import V100-optimized memory management functions from common module
from common.gpu_optimization import clear_gpu_cache, emergency_cleanup, cleanup_model_handler

print("💡 V100 memory management functions imported from common.gpu_optimization")
print("💡 Functions available:")
print("   - cleanup_model_handler(): Clean up model handlers before reinitializing")
print("   - clear_gpu_cache(): Clear GPU memory cache")
print("   - emergency_cleanup(): Aggressive cleanup for OOM recovery")

🔧 CUDA memory allocation configured: max_split_size_mb:64 (V100 optimized)
💡 V100 memory management functions imported from common.gpu_optimization
💡 Functions available:
   - cleanup_model_handler(): Clean up model handlers before reinitializing
   - clear_gpu_cache(): Clear GPU memory cache
   - emergency_cleanup(): Aggressive cleanup for OOM recovery


In [ ]:
# Enhanced imports - adding advanced analytics, visualization, and reporting
import os
import sys
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Any
from datetime import datetime
from IPython.display import Markdown, display

# Set project root (notebook is now in project root)
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Core InternVL3 imports (keep existing working imports)
from models.document_aware_internvl3_processor import DocumentAwareInternVL3Processor
from common.evaluation_metrics import load_ground_truth
from common.unified_schema import DocumentTypeFieldSchema

# NEW: Advanced analytics, visualization, and reporting imports
from common.batch_analytics import BatchAnalytics
from common.batch_visualizations import BatchVisualizer
from common.batch_reporting import BatchReporter
from common.document_type_metrics import DocumentTypeEvaluator

# Rich console for enhanced output
from rich import print as rprint
from rich.console import Console

# Initialize console for enhanced output
console = Console()

print("📚 ENHANCED LIBRARIES IMPORTED SUCCESSFULLY")
print("=" * 60)
print(f"🗂️ Project root: {project_root}")
print(f"📍 Current directory: {Path.cwd()}")
print("✅ Core InternVL3 imports: DocumentAwareInternVL3Processor, evaluation_metrics, unified_schema")
print("✅ NEW: BatchAnalytics - comprehensive DataFrames and statistics")
print("✅ NEW: BatchVisualizer - professional dashboards and heatmaps")
print("✅ NEW: BatchReporter - executive summaries and JSON reports")
print("✅ NEW: DocumentTypeEvaluator - ground truth evaluation")
print("🎯 Strategy: Direct processor approach (NO BatchProcessor - avoids recursion)")
print("🚀 Ready for enhanced batch processing with professional analytics!")

In [ ]:
from common.internvl3_batch_display import (
    display_prompt_info, 
    display_raw_and_cleaned, 
    display_field_comparison,
    display_image,
    display_complete_processing,
    set_display_config,
    get_display_config
)
from common.internvl3_batch_analytics import generate_summary_stats, display_summary_table, create_simple_dashboard

# Enhanced display configuration for full feature display
set_display_config(
    show_images=True,
    truncate_responses=False,  # Show full responses
    show_full_prompts=True,   # Show full prompts
    max_prompt_length=None,   # No truncation
    max_response_length=None, # No truncation  
    max_field_length=None,    # No truncation
    table_width=120,          # Wide tables like Llama
    use_syntax_highlighting=True,
    show_prompt_statistics=True
)

print("🔧 Enhanced display configuration applied:")
print("   ✅ Full prompts and responses (no truncation)")
print("   ✅ Image display enabled")
print("   ✅ 120-character wide tables") 
print("   ✅ Syntax highlighting enabled")
print("   ✅ Comprehensive field comparison like Llama version")

## Configuration

Set up paths and initialize the processor with **the exact same configuration** that works in the single-image notebook:

In [ ]:
# Enhanced Configuration System - Professional batch processing setup
import os
from datetime import datetime
from pathlib import Path

# Comprehensive configuration dictionary (replacing simple variables)
CONFIG = {
    # Model settings
    'MODEL_PATH': "/home/jovyan/nfs_share/models/InternVL3-8B",  # Jupyter environment path
    
    # Batch settings
    'DATA_DIR': 'evaluation_data',
    'GROUND_TRUTH': 'evaluation_data/ground_truth.csv',
    'MAX_IMAGES': None,  # None for all, or set limit
    'DOCUMENT_TYPES': None,  # None for all, or ['invoice', 'receipt']
    
    # Output settings
    'OUTPUT_BASE': os.getenv('OUTPUT_DIR', 'output'),
    'VERBOSE': True,
    
    # V100 optimization settings
    'USE_QUANTIZATION': True,
    'DEVICE_MAP': 'auto',
    'MAX_NEW_TOKENS': 4000,
    'TORCH_DTYPE': 'bfloat16',
    'LOW_CPU_MEM_USAGE': True
}

# Enhanced output directory structure
OUTPUT_BASE = Path(CONFIG['OUTPUT_BASE'])
if not OUTPUT_BASE.is_absolute():
    OUTPUT_BASE = Path.cwd() / OUTPUT_BASE

BATCH_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIRS = {
    'base': OUTPUT_BASE,
    'batch': OUTPUT_BASE / 'batch_results',
    'csv': OUTPUT_BASE / 'csv',
    'visualizations': OUTPUT_BASE / 'visualizations',
    'reports': OUTPUT_BASE / 'reports'
}

# Create all output directories
for dir_path in OUTPUT_DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

# Clean up any existing models before initializing
cleanup_model_handler('processor', globals())

print("🔧 ENHANCED CONFIGURATION LOADED")
print("=" * 50)
print(f"🎯 Model Path: {CONFIG['MODEL_PATH']}")
print(f"📁 Data Directory: {CONFIG['DATA_DIR']}")
print(f"📊 Ground Truth: {CONFIG['GROUND_TRUTH']}")
print(f"💾 Output Base: {OUTPUT_BASE}")
print(f"📁 Output Structure:")
for name, path in OUTPUT_DIRS.items():
    print(f"   {name}: {path}")
print(f"⏰ Batch Timestamp: {BATCH_TIMESTAMP}")
print("✅ All output directories created successfully")

## Step 1: Discover Images for Batch Processing

In [ ]:
# Enhanced image discovery using CONFIG settings
image_dir = project_root / CONFIG['DATA_DIR']
image_extensions = [".png", ".jpg", ".jpeg", ".bmp", ".tiff"]

# Find all images in directory
image_files = []
for ext in image_extensions:
    image_files.extend(list(image_dir.glob(f"*{ext}")))
    image_files.extend(list(image_dir.glob(f"*{ext.upper()}")))

image_files = sorted(list(set(image_files)))  # Remove duplicates and sort

# Apply CONFIG filters
if CONFIG['DOCUMENT_TYPES']:
    rprint(f"[cyan]📋 Filtering for document types: {CONFIG['DOCUMENT_TYPES']}[/cyan]")
    # Note: Document type filtering would require ground truth or filename patterns
    
if CONFIG['MAX_IMAGES']:
    original_count = len(image_files)
    image_files = image_files[:CONFIG['MAX_IMAGES']]
    rprint(f"[cyan]🔢 Limited to {CONFIG['MAX_IMAGES']} images (from {original_count} available)[/cyan]")

rprint(f"[bold green]📁 Discovered {len(image_files)} images in {image_dir}[/bold green]")
rprint("[cyan]📄 Image files:[/cyan]")
for i, img_path in enumerate(image_files[:10], 1):  # Show first 10
    rprint(f"   {i:2d}. {img_path.name}")
if len(image_files) > 10:
    rprint(f"   ... and {len(image_files) - 10} more images")

if len(image_files) == 0:
    rprint("[red]❌ No images found! Please check the CONFIG['DATA_DIR'] path.[/red]")
else:
    rprint(f"[green]✅ Ready to process {len(image_files)} images with enhanced analytics[/green]")

## Step 2: Initialize Document Schema

Set up the unified schema system for document-aware field selection:

In [ ]:
# Initialize document schema for field selection
print("📋 Initializing Document Type Schema...")

# Use model-specific schema like the working single-image notebook
schema_loader = DocumentTypeFieldSchema(model="internvl3")  # Specify internvl3 model

print(f"✅ Schema loaded with {schema_loader.total_fields} total fields")

# Get supported document types
supported_types = ["invoice", "receipt", "bank_statement"]  # Known types from YAML

print(f"📊 Document types supported: {', '.join(supported_types)}")

# Show field counts by document type
print("\n📈 Field counts by document type:")
for doc_type in supported_types:
    try:
        # Use correct method name with model context
        fields = schema_loader.get_document_fields(doc_type)
        print(f"   📄 {doc_type}: {len(fields)} fields")
    except Exception as e:
        print(f"   ❌ {doc_type}: Error getting fields - {e}")

# Step 3: Process Each Image in the Batch

Loop through each image, applying the same processing logic as in the single-image notebook:

In [ ]:
# Enhanced batch processing using modular EnhancedBatchProcessor
from common.enhanced_batch_processor import run_enhanced_batch_processing

# Run comprehensive batch processing with all enhanced features
batch_results, processing_times, document_types_found = run_enhanced_batch_processing(
    image_files=image_files,
    processor_class=DocumentAwareInternVL3Processor,
    schema_loader=schema_loader,
    config=CONFIG,
    show_enhanced_display=True  # Show full display features like Llama version
)

print(f"\n✅ Batch processing complete!")
print(f"📊 Results: {len(batch_results)} images processed")
print(f"⏱️ Times: {len(processing_times)} processing times recorded")
print(f"📋 Document types: {len(document_types_found)} types found")
print("🚀 Ready for enhanced analytics and reporting!")

## Step 4: Results Analysis and Export

Analyze the batch results and export to CSV:

In [ ]:
# Enhanced results analysis and CSV export using modular functions
from common.enhanced_batch_results import analyze_and_export_results

# Comprehensive analysis and export
analysis_results, csv_path, analytics_tuple = analyze_and_export_results(
    batch_results=batch_results,
    processing_times=processing_times,
    output_dirs=OUTPUT_DIRS,
    timestamp=BATCH_TIMESTAMP,
    config=CONFIG
)

# Extract components from analytics tuple
saved_files, df_results, df_summary, df_doctype_stats, df_field_stats = analytics_tuple

print(f"\n✅ Results analysis and export complete!")
print(f"📊 Analysis: {analysis_results['total_images']} images analyzed")
print(f"💾 CSV export: {csv_path}")
print(f"📈 Analytics: {len(saved_files)} DataFrames generated")
print("🚀 Ready for visualizations and reporting!")

## Step 5: Professional Analytics, Visualizations & Reporting

Generate comprehensive analytics, professional visualizations, executive reports, and final summary dashboard using the enhanced modular system:

In [ ]:
# Display enhanced analytics results in notebook format
from IPython.display import display

# Display comprehensive summary statistics
if len(df_results) > 0:
    print("📊 ENHANCED RESULTS SUMMARY")
    display(df_summary)
else:
    print("⚠️ No results available for summary")

# Display document type performance if available
if not df_doctype_stats.empty:
    print("\n📋 DOCUMENT TYPE PERFORMANCE")
    display(df_doctype_stats)
else:
    print("⚠️ No document type statistics available")

# Display field-level statistics if available
if df_field_stats is not None and not df_field_stats.empty:
    print("\n🎯 FIELD-LEVEL ACCURACY STATISTICS")
    print("Top 10 best performing fields:")
    display(df_field_stats.head(10))
else:
    print("⚠️ No field-level accuracy data available")

print("\n✅ Enhanced analytics display complete!")
print("🚀 Ready for professional visualizations and comprehensive reporting!")

In [ ]:
# Comprehensive summary with visualizations, reports, and final dashboard
from common.enhanced_batch_summary import generate_complete_summary

# Generate complete summary with all professional features
viz_files, report_files, summary_data, summary_report_path = generate_complete_summary(
    batch_results=batch_results,
    processing_times=processing_times,
    document_types_found=document_types_found,
    output_dirs=OUTPUT_DIRS,
    timestamp=BATCH_TIMESTAMP,
    config=CONFIG,
    analytics_tuple=analytics_tuple,
    show_visualizations=True  # Display visualizations in notebook
)

print(f"\n🎉 COMPREHENSIVE ENHANCED PROCESSING COMPLETE!")
print(f"📊 Summary data: {summary_data['total_images']} images processed")
print(f"🎨 Visualizations: {len(viz_files) if viz_files else 0} files generated")
print(f"📄 Reports: {len(report_files) if report_files else 0} files generated") 
print(f"📝 Summary report: {summary_report_path}")
print("✅ InternVL3 now has complete feature parity with Llama version!")